# 💳 Scoring de Crédito — Clasificación de Riesgo Fintech

**Dataset:** Give Me Some Credit (Kaggle) — 150,000 prestatarios reales  
**Objetivo:** Predecir la probabilidad de que un cliente incurra en mora ≥90 días en los próximos 2 años

**Pipeline completo:**
1. EDA orientado al problema de negocio
2. Manejo de valores nulos y outliers
3. Feature engineering financiero
4. Modelos: Logistic Regression vs Random Forest
5. Métricas orientadas a negocio (no solo accuracy)
6. Análisis de umbral óptimo según costo de error

**Relevancia para fintech LATAM:** El scoring crediticio es el modelo más crítico en cualquier empresa de préstamos digitales (Yape, Tigo Money, neobancos). Un mal modelo cuesta dinero real.

In [ ]:
# ─── DEPENDENCIAS ───────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    roc_auc_score, roc_curve, classification_report,
    confusion_matrix, precision_recall_curve, average_precision_score
)
from sklearn.calibration import calibration_curve
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 11
SEED = 42

print('✅ Librerías cargadas')

## 1. Carga y Primera Inspección

In [ ]:
# ─── CARGA ───────────────────────────────────────────────────────────────────
# Descarga desde: https://www.kaggle.com/c/GiveMeSomeCredit/data
# Archivo: cs-training.csv
df = pd.read_csv('datos/cs-training.csv', index_col=0)

# Renombrar columnas a español para claridad
COLUMNAS = {
    'SeriousDlqin2yrs':                     'target',          # 1 = mora >= 90 días
    'RevolvingUtilizationOfUnsecuredLines':  'util_credito',    # % utilizado de líneas revolving
    'age':                                   'edad',
    'NumberOfTime30-59DaysPastDueNotWorse':  'mora_30_59',
    'DebtRatio':                             'ratio_deuda',
    'MonthlyIncome':                         'ingreso_mensual',
    'NumberOfOpenCreditLinesAndLoans':       'lineas_credito',
    'NumberOfTimes90DaysLate':               'mora_90',
    'NumberRealEstateLoansOrLines':          'prestamos_inmuebles',
    'NumberOfTime60-89DaysPastDueNotWorse':  'mora_60_89',
    'NumberOfDependents':                    'dependientes',
}
df.rename(columns=COLUMNAS, inplace=True)

print(f'Shape: {df.shape}')
print(f'\nDistribución del target:')
print(df['target'].value_counts())
print(f'\nTasa de mora: {df["target"].mean()*100:.2f}%')
print(f'\n⚠️  Dataset DESBALANCEADO — la clase minoritaria (mora) representa solo ~6.7%')

## 2. EDA — Análisis Exploratorio

In [ ]:
# ─── VALORES NULOS ────────────────────────────────────────────────────────────
nulos = df.isnull().sum()
nulos_pct = (nulos / len(df) * 100).round(2)
print('Valores nulos por columna:')
print(pd.DataFrame({'Nulos': nulos, 'Porcentaje %': nulos_pct})[nulos > 0])

print('\n💡 ingreso_mensual tiene ~29% nulos → imputar con mediana por grupo de edad')
print('   dependientes tiene ~2.6% nulos → imputar con mediana global')

In [ ]:
# ─── ANÁLISIS DE VARIABLES VS TARGET ─────────────────────────────────────────
features_num = ['util_credito', 'edad', 'ratio_deuda', 'ingreso_mensual',
                'lineas_credito', 'mora_30_59', 'mora_60_89', 'mora_90']

fig, axes = plt.subplots(2, 4, figsize=(18, 10))
axes = axes.flatten()

for i, col in enumerate(features_num):
    ax = axes[i]
    # Distribución por target usando violinplot
    temp = df[[col, 'target']].dropna()
    temp_clip = temp.copy()
    # Clipear outliers extremos para visualización
    p99 = temp[col].quantile(0.99)
    temp_clip[col] = temp[col].clip(upper=p99)
    
    ax.violinplot(
        [temp_clip[temp_clip['target']==0][col].values,
         temp_clip[temp_clip['target']==1][col].values],
        positions=[0, 1], showmedians=True
    )
    ax.set_xticks([0, 1])
    ax.set_xticklabels(['Sin mora', 'Con mora'])
    ax.set_title(col, fontsize=10)

plt.suptitle('Distribución de variables por clase (target)', fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('img/01_eda_variables.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Preprocesamiento y Feature Engineering

In [ ]:
# ─── LIMPIEZA Y FEATURE ENGINEERING ──────────────────────────────────────────

df_clean = df.copy()

# 1. Imputar ingreso_mensual con mediana por grupo de edad
df_clean['edad_grupo'] = pd.cut(df_clean['edad'], bins=[0,25,35,45,55,65,120], labels=False)
mediana_ingreso = df_clean.groupby('edad_grupo')['ingreso_mensual'].transform('median')
df_clean['ingreso_mensual'] = df_clean['ingreso_mensual'].fillna(mediana_ingreso)
df_clean['ingreso_mensual'] = df_clean['ingreso_mensual'].fillna(df_clean['ingreso_mensual'].median())

# 2. Imputar dependientes con mediana global
df_clean['dependientes'] = df_clean['dependientes'].fillna(df_clean['dependientes'].median())

# 3. Tratamiento de outliers en utilización de crédito
#    Valores > 1 son datos erróneos (% no puede superar 100%)
df_clean['util_credito'] = df_clean['util_credito'].clip(0, 1)

# 4. Feature engineering financiero
df_clean['total_moras']      = df_clean['mora_30_59'] + df_clean['mora_60_89'] + df_clean['mora_90']
df_clean['ingreso_por_linea'] = df_clean['ingreso_mensual'] / (df_clean['lineas_credito'] + 1)
df_clean['carga_deuda']      = (df_clean['ratio_deuda'] * df_clean['ingreso_mensual']).clip(upper=1e6)
df_clean['es_joven']         = (df_clean['edad'] < 30).astype(int)
df_clean['tiene_dependientes'] = (df_clean['dependientes'] > 0).astype(int)

print('✅ Preprocesamiento completado')
print(f'   Nulos restantes: {df_clean.isnull().sum().sum()}')

FEATURES = [
    'util_credito', 'edad', 'ratio_deuda', 'ingreso_mensual',
    'lineas_credito', 'mora_30_59', 'mora_60_89', 'mora_90',
    'prestamos_inmuebles', 'dependientes',
    'total_moras', 'ingreso_por_linea', 'carga_deuda',  # Features nuevos
    'es_joven', 'tiene_dependientes'
]

X = df_clean[FEATURES]
y = df_clean['target']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)
print(f'\n   Train: {X_train.shape[0]:,} | Test: {X_test.shape[0]:,}')
print(f'   Tasa mora train: {y_train.mean()*100:.2f}%  |  test: {y_test.mean()*100:.2f}%')

## 4. Modelos de Clasificación

In [ ]:
# ─── ENTRENAMIENTO ────────────────────────────────────────────────────────────
# Pipeline con imputación (por si quedan nulos) + escalado + modelo

modelos = {
    'Logistic Regression': Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler',  StandardScaler()),
        ('model',   LogisticRegression(class_weight='balanced', max_iter=1000, random_state=SEED))
    ]),
    'Random Forest': Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('model',   RandomForestClassifier(
            n_estimators=200, max_depth=8, min_samples_leaf=50,
            class_weight='balanced', random_state=SEED, n_jobs=-1
        ))
    ]),
}

resultados = {}
for nombre, pipeline in modelos.items():
    print(f'Entrenando {nombre}...')
    pipeline.fit(X_train, y_train)
    proba = pipeline.predict_proba(X_test)[:, 1]
    auc   = roc_auc_score(y_test, proba)
    ap    = average_precision_score(y_test, proba)
    resultados[nombre] = {'pipeline': pipeline, 'proba': proba, 'auc': auc, 'ap': ap}
    print(f'   AUC-ROC: {auc:.4f}  |  Avg Precision: {ap:.4f}')

mejor_modelo_nombre = max(resultados, key=lambda k: resultados[k]['auc'])
print(f'\n🏆 Mejor modelo: {mejor_modelo_nombre}')

In [ ]:
# ─── CURVAS ROC Y PRECISION-RECALL ────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

colores = ['#2E86C1', '#E74C3C']
for (nombre, res), color in zip(resultados.items(), colores):
    # ROC
    fpr, tpr, _ = roc_curve(y_test, res['proba'])
    axes[0].plot(fpr, tpr, color=color, lw=2, label=f'{nombre} (AUC={res["auc"]:.3f})')

    # Precision-Recall
    prec, rec, _ = precision_recall_curve(y_test, res['proba'])
    axes[1].plot(rec, prec, color=color, lw=2, label=f'{nombre} (AP={res["ap"]:.3f})')

# ROC
axes[0].plot([0,1],[0,1],'k--', lw=1, label='Aleatorio (AUC=0.5)')
axes[0].set_xlabel('Tasa Falsos Positivos'); axes[0].set_ylabel('Tasa Verdaderos Positivos')
axes[0].set_title('Curva ROC', fontweight='bold'); axes[0].legend()

# P-R
baseline = y_test.mean()
axes[1].axhline(baseline, color='k', linestyle='--', lw=1, label=f'Baseline ({baseline:.3f})')
axes[1].set_xlabel('Recall'); axes[1].set_ylabel('Precision')
axes[1].set_title('Curva Precision-Recall\n(más informativa con clases desbalanceadas)', fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.savefig('img/02_curvas_roc_pr.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Umbral Óptimo — Decisión de Negocio

In [ ]:
# ─── ANÁLISIS DE UMBRAL ────────────────────────────────────────────────────────
# En crédito, los errores NO tienen el mismo costo:
#   Falso Negativo (FN): aprobar a alguien que hará mora → COSTO ALTO (pérdida del capital)
#   Falso Positivo (FP): rechazar a alguien que pagaría → COSTO BAJO (lucro cesante)
#
# Asumimos: costo FN = 5x costo FP (perder el préstamo es 5x peor que perder un cliente)

COSTO_FN = 5  # Pérdida por aprobar un moroso
COSTO_FP = 1  # Costo por rechazar un buen cliente

mejor = resultados[mejor_modelo_nombre]
umbrales = np.arange(0.01, 0.99, 0.01)
costos = []

for umbral in umbrales:
    pred = (mejor['proba'] >= umbral).astype(int)
    cm   = confusion_matrix(y_test, pred)
    tn, fp, fn, tp = cm.ravel()
    costo = fn * COSTO_FN + fp * COSTO_FP
    costos.append(costo)

umbral_optimo = umbrales[np.argmin(costos)]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(umbrales, costos, color='#E74C3C', lw=2)
axes[0].axvline(umbral_optimo, color='#2E86C1', linestyle='--', lw=2,
                label=f'Umbral óptimo: {umbral_optimo:.2f}')
axes[0].set_xlabel('Umbral de clasificación')
axes[0].set_ylabel('Costo total (FN×5 + FP×1)')
axes[0].set_title('Costo total por umbral\n(FN tiene 5x el costo de FP)', fontweight='bold')
axes[0].legend()

# Matriz de confusión con umbral óptimo
pred_opt = (mejor['proba'] >= umbral_optimo).astype(int)
cm_opt   = confusion_matrix(y_test, pred_opt)
sns.heatmap(cm_opt, annot=True, fmt='d', cmap='Blues', ax=axes[1],
            xticklabels=['Predice: Paga', 'Predice: Mora'],
            yticklabels=['Real: Paga', 'Real: Mora'])
axes[1].set_title(f'Matriz de Confusión\nUmbral = {umbral_optimo:.2f}', fontweight='bold')

plt.tight_layout()
plt.savefig('img/03_umbral_optimo.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\n🎯 Umbral óptimo: {umbral_optimo:.2f}')
print(f'\n{classification_report(y_test, pred_opt, target_names=["Paga", "Mora"])}')

## 6. Feature Importance

In [ ]:
# ─── IMPORTANCIA DE VARIABLES (Random Forest) ─────────────────────────────────
rf_pipeline = resultados['Random Forest']['pipeline']
importancias = rf_pipeline.named_steps['model'].feature_importances_

fi_df = pd.DataFrame({
    'Feature':     FEATURES,
    'Importancia': importancias
}).sort_values('Importancia', ascending=True)

fig, ax = plt.subplots(figsize=(10, 7))
bars = ax.barh(fi_df['Feature'], fi_df['Importancia'], color='#2E86C1', alpha=0.85)

# Colorear features de ingeniería en naranja
features_nuevos = ['total_moras', 'ingreso_por_linea', 'carga_deuda', 'es_joven', 'tiene_dependientes']
for bar, feat in zip(bars, fi_df['Feature']):
    if feat in features_nuevos:
        bar.set_color('#E67E22')

ax.set_xlabel('Importancia relativa')
ax.set_title('Feature Importance — Random Forest\n(naranja = features de ingeniería)', fontweight='bold')
ax.axvline(importancias.mean(), color='red', linestyle='--', alpha=0.7, label='Media')
ax.legend()
plt.tight_layout()
plt.savefig('img/04_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print('💡 Las variables más predictivas de mora:')
for _, row in fi_df.tail(5).iloc[::-1].iterrows():
    print(f'   {row["Feature"]:<25} {row["Importancia"]:.4f}')

## 7. Conclusiones

In [ ]:
# ─── RESUMEN ─────────────────────────────────────────────────────────────────
print('=' * 60)
print('RESUMEN EJECUTIVO — SCORING DE CRÉDITO')
print('=' * 60)
for nombre, res in resultados.items():
    print(f'\n{nombre}:')
    print(f'   AUC-ROC:           {res["auc"]:.4f}')
    print(f'   Average Precision: {res["ap"]:.4f}')

print(f'\n🏆 Modelo ganador: {mejor_modelo_nombre}')
print(f'   Umbral óptimo (costo FN=5x FP=1): {umbral_optimo:.2f}')
print(f'\n💡 Insights clave:')
print(f'   1. Dataset desbalanceado (6.7% mora) → usar AUC y AP, NO accuracy')
print(f'   2. El umbral óptimo ≠ 0.5 cuando los costos de error son asimétricos')
print(f'   3. Features de ingeniería (total_moras, ingreso_por_linea) mejoran el modelo')
print(f'   4. Random Forest supera a Regresión Logística en AUC pero es menos interpretable')